# Orientation Selectivity in Mouse Visual Cortex

This notebook demonstrates **orientation selectivity**, a hallmark property of neurons
in primary and higher visual cortical areas, using extracellular Neuropixels recordings
from the [Allen Institute Visual Coding — Neuropixels](https://dandiarchive.org/dandiset/000021)
dataset (DANDI:000021, Brain Observatory 1.1 stimulus set).

During each recording session, a mouse passively viewed drifting sinusoidal gratings
presented at 8 directions of motion (0-315 deg in 45 deg steps) x 5 temporal frequencies
(1, 2, 4, 8, 15 Hz), each repeated ~15 times, interleaved with blank (gray screen) sweeps.
We stream a single session directly from DANDI (no local download), isolate well-isolated
("good" quality) units recorded in visual cortical areas (VISp, VISl, VISal, VISrl, VISam,
VIS), and compute each neuron's firing-rate tuning curve as a function of grating direction.
Orientation selectivity is quantified with the standard circular-variance based global
orientation selectivity index (gOSI), and tuning significance is assessed with a one-way
ANOVA across directions computed from single-trial firing rates.

## Setup and Data Loading

In [1]:
import time

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pynapple as nap
import remfile
from dandi.dandiapi import DandiAPIClient
from matplotlib.gridspec import GridSpec
from pynwb import NWBHDF5IO
from scipy import stats
from tqdm import tqdm

nap.nap_config.suppress_conversion_warnings = True
np.random.seed(0)

DANDISET_ID = "000021"
SESSION_PATH = "sub-707296975/sub-707296975_ses-721123822.nwb"

with DandiAPIClient() as client:
    asset = client.get_dandiset(DANDISET_ID).get_asset_by_path(SESSION_PATH)
    s3_url = asset.get_content_url(follow_redirects=1, strip_query=True)

print("Streaming:", s3_url)

disk_cache = remfile.DiskCache("/tmp/remfile_cache")
rem_file = remfile.File(s3_url, disk_cache=disk_cache)
h5py_file = h5py.File(rem_file, "r")
io = NWBHDF5IO(file=h5py_file)
nwbfile = io.read()
print(f"Session: {nwbfile.session_id}  |  subject: {nwbfile.subject.subject_id}  |  "
      f"genotype: {nwbfile.subject.genotype}")

Streaming: https://dandiarchive.s3.amazonaws.com/blobs/9f2/c60/9f2c6063-0bcf-4c5e-9039-c6d7144bcbe1


Session: 721123822  |  subject: 707296975  |  genotype: Pvalb-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt


## Inspect NWB Structure

The full session NWB file bundles the spike-sorted `units` table together with the
stimulus presentation tables (`intervals`). We avoid `units.to_dataframe()` here because
it eagerly pulls every ragged column (spike_times, spike_amplitudes, waveform_mean) for
**all** units in the file (>1600 in this session) over the network; instead we read only
the lightweight scalar metadata columns first, decide which units we want, and then fetch
spike times for just those units.

In [2]:
print("Acquisition:", list(nwbfile.acquisition.keys()))
print("Intervals (stimulus tables):", list(nwbfile.intervals.keys()))
print("Processing modules:", list(nwbfile.processing.keys()))
print("Total units in file:", len(nwbfile.units))

Acquisition: ['raw_running_wheel_rotation', 'running_wheel_signal_voltage', 'running_wheel_supply_voltage']
Intervals (stimulus tables): ['drifting_gratings_presentations', 'flashes_presentations', 'gabors_presentations', 'invalid_times', 'natural_movie_one_presentations', 'natural_movie_three_presentations', 'natural_scenes_presentations', 'spontaneous_presentations', 'static_gratings_presentations']
Processing modules: ['eye_tracking_rig_metadata', 'optotagging', 'running', 'stimulus']
Total units in file: 1603


In [3]:
units_tbl = nwbfile.units
light_cols = ["quality", "peak_channel_id", "snr", "firing_rate", "isi_violations",
              "presence_ratio", "amplitude_cutoff"]
units_meta = pd.DataFrame({c: units_tbl[c].data[:] for c in light_cols})

electrodes = nwbfile.electrodes.to_dataframe()
units_meta["location"] = units_meta["peak_channel_id"].map(electrodes["location"])

print(units_meta["quality"].value_counts())
print()
print("Unit count by brain area (quality == 'good'):")
print(units_meta.loc[units_meta["quality"] == "good", "location"].value_counts())

quality
good     1191
noise     412
Name: count, dtype: int64

Unit count by brain area (quality == 'good'):
location
CA1      194
LP       180
VISal    139
VISp     110
VIS       97
VISl      95
VISam     92
VISrl     89
DG        82
LGv       33
APN       23
SCig      19
CA3       16
MB        11
POL        7
HPF        3
LGd        1
Name: count, dtype: int64


We keep well-isolated units (`quality == "good"`) located in visual cortex (areas whose
acronym starts with `VIS`: VISp = primary visual cortex, VISl, VISal, VISrl, VISam =
higher visual areas, and `VIS` = unassigned visual cortex).

In [4]:
is_good_vis = (units_meta["quality"] == "good") & (
    units_meta["location"].astype(str).str.startswith("VIS")
)
unit_indices = units_meta.index[is_good_vis].to_numpy()
print(f"Selected {len(unit_indices)} good visual-cortex units")

spike_times_dict = {}
for idx in tqdm(unit_indices, desc="Fetching spike times"):
    spike_times_dict[int(idx)] = np.asarray(units_tbl["spike_times"][int(idx)])

spikes = nap.TsGroup(
    {idx: nap.Ts(t=t) for idx, t in spike_times_dict.items()},
    location=units_meta.loc[unit_indices, "location"],
    snr=units_meta.loc[unit_indices, "snr"],
)
print(spikes)

Selected 622 good visual-cortex units


Fetching spike times:   0%|          | 0/622 [00:00<?, ?it/s]

Fetching spike times:  22%|██▏       | 135/622 [00:00<00:00, 1344.15it/s]

Fetching spike times:  43%|████▎     | 270/622 [00:00<00:00, 1186.82it/s]

Fetching spike times:  66%|██████▌   | 411/622 [00:00<00:00, 1278.79it/s]

Fetching spike times:  90%|█████████ | 562/622 [00:00<00:00, 1363.91it/s]

Fetching spike times: 100%|██████████| 622/622 [00:00<00:00, 1314.76it/s]

  Index      rate  location      snr
-------  --------  ----------  -----
     86   5.50077  VISam        2.39
     87   3.23201  VISam        2.24
     89   3.0573   VISam        1.9
     90   2.18651  VISam        2.21
     91   0.74949  VISam        3.24
     92   3.17697  VISam        3.79
     93   2.41595  VISam        2.11
     94  13.0149   VISam        3.48
     95   0.21171  VISam        4.52
     96   5.06298  VISam        0.98
     97   0.41934  VISam        4.17
     98   1.051    VISam        2.98
     99   5.77476  VISam        2.7
    100   0.65908  VISam        1.75
    101   1.14315  VISam        2.08
    102   0.36532  VISam        1.89
    103   3.39438  VISam        2.52
    104   5.91716  VISam        2.51
    105   2.59087  VISam        2.71
    106   3.47114  VISam        2.16
    107   4.91845  VISam        1.1
    108   5.03454  VISam        1.84
    109   6.57379  VISam        0.97
    110   0.42209  VISam        4.76
    111   7.02453  VISam        4.11
    

/Users/bdichter/miniconda3/lib/python3.12/site-packages/pynapple/core/ts_group.py:318: FutureWarning: initializing metadata with variable keyword arguments may be unsupported in a future version of Pynapple. Instead, initialize using the metadata argument.
  warnings.warn(


## Raw Data Validation

Before any trial-based analysis, plot raw spike times for a handful of units across the
whole session, together with the drifting-gratings stimulus epoch, to confirm the data
stream is sane (continuous recording, reasonable firing rates, stimulus block visible).

In [5]:
dg_all = nwbfile.intervals["drifting_gratings_presentations"].to_dataframe()
stim_start, stim_stop = dg_all["start_time"].min(), dg_all["stop_time"].max()

example_units_raw = list(spikes.keys())[:15]
fig, ax = plt.subplots(figsize=(12, 5))
for row, uid in enumerate(example_units_raw):
    t = spike_times_dict[uid]
    ax.plot(t, np.full(len(t), row), "|", color="k", markersize=2, alpha=0.6)
ax.axvspan(stim_start, stim_stop, color="tab:orange", alpha=0.15,
           label="drifting gratings block")
ax.set_xlabel("Session time (s)")
ax.set_ylabel("Unit (arbitrary order)")
ax.set_yticks(range(len(example_units_raw)))
ax.set_yticklabels(example_units_raw)
ax.set_title("Raw spike trains, 15 example visual-cortex units, full session")
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig("fig01_raw_spike_trains.png", dpi=150)
plt.close()

## Building Trial Intervals and Per-Trial Firing Rates

The `drifting_gratings_presentations` table has one row per stimulus presentation
(2 s grating drift), with `orientation` (0-315 deg, the *direction* of grating motion)
and `temporal_frequency` columns; 30 rows have `orientation == NaN` and correspond to
blank (0-contrast) sweeps, which we exclude from the tuning curve trials.

In [6]:
dg_stim = dg_all.dropna(subset=["orientation"]).reset_index(drop=True)
print(f"{len(dg_stim)} stimulus trials across "
      f"{dg_stim['orientation'].nunique()} directions x "
      f"{dg_stim['temporal_frequency'].nunique()} temporal frequencies")

trials = nap.IntervalSet(start=dg_stim["start_time"].values, end=dg_stim["stop_time"].values)
durations = (dg_stim["stop_time"] - dg_stim["start_time"]).values

counts = spikes.count(ep=trials)  # TsdFrame, one row per trial, one column per unit
trial_rates = pd.DataFrame(counts.values / durations[:, None], columns=list(spikes.keys()))
trial_rates["orientation"] = dg_stim["orientation"].values
trial_rates["temporal_frequency"] = dg_stim["temporal_frequency"].values

unit_ids = list(spikes.keys())
tuning = trial_rates.groupby("orientation")[unit_ids].mean()  # (8 directions, n_units)
print(tuning.shape)

598 stimulus trials across 8 directions x 5 temporal frequencies
(8, 622)


## Orientation Selectivity Metrics

For each unit we compute the tuning curve $R(\theta)$ averaged over temporal frequency and
repeats, then the standard vector-sum **global orientation selectivity index (gOSI)**,
which uses the double angle to make it invariant to the 180 deg ambiguity of orientation
(a grating drifting at $\theta$ and $\theta+180$ deg has the same orientation):

$$\text{gOSI} = \frac{\left| \sum_\theta R(\theta) e^{i2\theta} \right|}{\sum_\theta R(\theta)}$$

and the analogous direction selectivity index (gDSI, single angle, sensitive to the 180 deg
ambiguity). Preferred orientation is the angle (mod 180 deg) of the gOSI vector. Tuning
significance is tested with a one-way ANOVA across the 8 directions using single-trial
(not averaged) firing rates.

In [7]:
orientations_deg = tuning.index.values.astype(float)
orientations_rad = np.deg2rad(orientations_deg)
R = tuning.values

num_osi = np.sum(R * np.exp(1j * 2 * orientations_rad)[:, None], axis=0)
num_dsi = np.sum(R * np.exp(1j * orientations_rad)[:, None], axis=0)
denom = np.sum(R, axis=0)

gOSI = np.abs(num_osi) / denom
gDSI = np.abs(num_dsi) / denom
pref_orientation_deg = np.rad2deg((np.angle(num_osi) / 2) % np.pi)
peak_rate_hz = R.max(axis=0)

pvals = np.array([
    stats.f_oneway(*[trial_rates.loc[trial_rates["orientation"] == o, uid].values
                      for o in orientations_deg])[1]
    for uid in unit_ids
])

metrics = pd.DataFrame({
    "unit_id": unit_ids,
    "location": units_meta.loc[unit_ids, "location"].values,
    "gOSI": gOSI,
    "gDSI": gDSI,
    "pref_orientation_deg": pref_orientation_deg,
    "peak_rate_hz": peak_rate_hz,
    "mean_rate_hz": denom / len(orientations_deg),
    "anova_p": pvals,
}).set_index("unit_id")

metrics["significant"] = metrics["anova_p"] < 0.01
print(f"{metrics['significant'].sum()} / {len(metrics)} units significantly direction/"
      f"orientation-tuned (one-way ANOVA, p < 0.01)")
print(metrics.groupby("location")["gOSI"].median().sort_values(ascending=False))

metrics.to_csv("unit_orientation_metrics.csv")
tuning.to_csv("tuning_curves_by_orientation.csv")

/var/folders/67/qdwczmzx315gj1xp7hp1f11r0000gn/T/ipykernel_43819/3864798697.py:9: RuntimeWarning: invalid value encountered in divide
  gOSI = np.abs(num_osi) / denom
/var/folders/67/qdwczmzx315gj1xp7hp1f11r0000gn/T/ipykernel_43819/3864798697.py:10: RuntimeWarning: invalid value encountered in divide
  gDSI = np.abs(num_dsi) / denom
/Users/bdichter/miniconda3/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:579: ConstantInputWarning: Each of the input arrays is constant; the F statistic is not defined or infinite
  res = hypotest_fun_out(*samples, **kwds)


455 / 622 units significantly direction/orientation-tuned (one-way ANOVA, p < 0.01)
location
VISl     0.202682
VISp     0.201042
VISal    0.193168
VISam    0.184244
VISrl    0.167885
VIS      0.163320
Name: gOSI, dtype: float64


## Example Neuron: Raster and Direction Tuning

We pick the visual-cortex unit with the highest gOSI (among significantly tuned units
with a peak firing rate above 2 Hz, to avoid picking a near-silent noisy unit) and show
its trial raster sorted by stimulus direction, followed by its polar tuning curve.

In [8]:
candidates = metrics[metrics["significant"] & (metrics["peak_rate_hz"] > 2)]
example_unit = candidates["gOSI"].idxmax()
print(f"Example unit: {example_unit}  (area={metrics.loc[example_unit, 'location']}, "
      f"gOSI={metrics.loc[example_unit, 'gOSI']:.2f})")

st_example = nap.Ts(t=spike_times_dict[example_unit])
onsets = nap.Ts(t=dg_stim["start_time"].values)
peri = nap.compute_perievent(st_example, onsets, window=(-0.5, 2.5))

order = dg_stim.sort_values("orientation").index.values
orientations_sorted = dg_stim.loc[order, "orientation"].values
cmap = plt.get_cmap("hsv")
ori_colors = {o: cmap(i / 8) for i, o in enumerate(np.unique(orientations_sorted))}

fig, axes = plt.subplots(1, 2, figsize=(13, 6),
                          gridspec_kw={"width_ratios": [1.6, 1]},
                          subplot_kw={"projection": None})
ax = axes[0]
for row, trial_idx in enumerate(order):
    spk = peri[trial_idx]
    color = ori_colors[orientations_sorted[row]]
    ax.plot(spk.index, np.full(len(spk), row), "|", color=color, markersize=3)
ax.axvline(0, color="k", linestyle="--", lw=1)
ax.axvline(2.0, color="k", linestyle="--", lw=1)
ax.set_xlabel("Time from stimulus onset (s)")
ax.set_ylabel("Trial (sorted by direction)")
ax.set_title(f"Unit {example_unit} ({metrics.loc[example_unit, 'location']}) raster,\n"
             f"sorted by grating direction")

uniq_ori = np.unique(orientations_sorted)
for o in uniq_ori:
    b = np.searchsorted(orientations_sorted, o)
    ax.axhline(b, color="gray", lw=0.5, alpha=0.4)
    ax.text(2.65, b + len(order) / 32, f"{int(o)} deg", fontsize=8, va="center",
            color=ori_colors[o])
ax.set_xlim(-0.5, 3.2)

axes[1].remove()
ax2 = fig.add_subplot(1, 2, 2, projection="polar")
theta_plot = np.append(orientations_rad, orientations_rad[0])
r_plot = np.append(R[:, unit_ids.index(example_unit)], R[0, unit_ids.index(example_unit)])
ax2.plot(theta_plot, r_plot, "-o", color="tab:blue")
ax2.fill(theta_plot, r_plot, alpha=0.2, color="tab:blue")
ax2.set_title(f"Direction tuning curve\ngOSI={metrics.loc[example_unit, 'gOSI']:.2f}, "
              f"pref. orientation={metrics.loc[example_unit, 'pref_orientation_deg']:.0f} deg",
              pad=20)
ax2.set_theta_zero_location("E")
ax2.set_theta_direction(1)

plt.tight_layout()
plt.savefig("fig02_example_unit_raster_and_tuning.png", dpi=150)
plt.close()

Example unit: 837  (area=VISl, gOSI=0.99)


## PSTHs by Direction for the Example Neuron

Peri-stimulus time histograms (50 ms bins, averaged over repeats) for each of the 8
grating directions show clear, direction-selective transient and sustained firing.

In [9]:
bin_edges = np.arange(-0.5, 2.51, 0.05)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

fig, ax = plt.subplots(figsize=(9, 5))
for o in uniq_ori:
    trial_idxs = dg_stim.index[dg_stim["orientation"] == o].to_numpy()
    all_rel_times = np.concatenate([peri[i].index.values for i in trial_idxs]) \
        if len(trial_idxs) else np.array([])
    counts_h, _ = np.histogram(all_rel_times, bins=bin_edges)
    rate_h = counts_h / (len(trial_idxs) * (bin_edges[1] - bin_edges[0]))
    ax.plot(bin_centers, rate_h, color=ori_colors[o], label=f"{int(o)} deg", lw=1.5)

ax.axvspan(0, 2.0, color="gray", alpha=0.1)
ax.set_xlabel("Time from stimulus onset (s)")
ax.set_ylabel("Firing rate (Hz)")
ax.set_title(f"Unit {example_unit}: PSTH by grating direction")
ax.legend(loc="upper right", fontsize=8, ncol=2, title="direction")
plt.tight_layout()
plt.savefig("fig03_example_unit_psth_by_direction.png", dpi=150)
plt.close()

## Population Summary

Four panels: (1) example polar tuning curves across several visual areas, (2) the gOSI
distribution for significantly tuned units, (3) tuning curves for all significantly tuned
units normalized to their peak and sorted by preferred orientation (classic "tuning matrix"
visualization), and (4) median gOSI by visual area.

In [10]:
sig = metrics[metrics["significant"] & (metrics["peak_rate_hz"] > 1)]
areas_present = [a for a in ["VISp", "VISl", "VISal", "VISrl", "VISam", "VIS"]
                  if a in sig["location"].unique()]
example_per_area = {
    a: sig[sig["location"] == a]["gOSI"].idxmax() for a in areas_present
}

fig = plt.figure(figsize=(15, 11))
gs = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.4)

n_ex = len(example_per_area)
gs_polar = gs[0, 0].subgridspec(2, 3, hspace=0.6, wspace=0.6)
for i, (area, uid) in enumerate(example_per_area.items()):
    r_ax = fig.add_subplot(gs_polar[i // 3, i % 3], projection="polar")
    r = R[:, unit_ids.index(uid)]
    r_plot = np.append(r, r[0])
    r_ax.plot(theta_plot, r_plot, "-o", color="tab:blue", markersize=3, lw=1)
    r_ax.fill(theta_plot, r_plot, alpha=0.2, color="tab:blue")
    r_ax.set_title(f"{area}\ngOSI={metrics.loc[uid, 'gOSI']:.2f}", fontsize=9, pad=12)
    r_ax.set_xticklabels([])
    r_ax.set_yticklabels([])
fig.text(0.16, 0.93, "Example direction tuning curves by area", ha="center", fontsize=11)

ax_hist = fig.add_subplot(gs[0, 1])
for area in areas_present:
    vals = sig.loc[sig["location"] == area, "gOSI"]
    ax_hist.hist(vals, bins=np.linspace(0, 1, 21), alpha=0.5, label=area, density=True)
ax_hist.set_xlabel("gOSI")
ax_hist.set_ylabel("Density")
ax_hist.set_title(f"gOSI distribution\n(n={len(sig)} significantly tuned units)")
ax_hist.legend(fontsize=7)

ax_bar = fig.add_subplot(gs[0, 2])
med = sig.groupby("location")["gOSI"].median().reindex(areas_present)
sem = sig.groupby("location")["gOSI"].sem().reindex(areas_present)
ax_bar.bar(med.index, med.values, yerr=sem.values, color="tab:blue", alpha=0.7, capsize=4)
ax_bar.set_ylabel("Median gOSI (+/- SEM)")
ax_bar.set_title("Orientation selectivity by area")
ax_bar.tick_params(axis="x", rotation=45)

ax_mat = fig.add_subplot(gs[1, :])
sig_sorted = sig.sort_values("pref_orientation_deg")
norm_curves = np.array([
    R[:, unit_ids.index(uid)] / R[:, unit_ids.index(uid)].max()
    for uid in sig_sorted.index
])
im = ax_mat.imshow(norm_curves, aspect="auto", cmap="viridis",
                    extent=[orientations_deg.min() - 22.5, orientations_deg.max() + 22.5,
                            len(norm_curves), 0])
ax_mat.set_xlabel("Grating direction (deg)")
ax_mat.set_ylabel("Unit (sorted by preferred orientation)")
ax_mat.set_title("Normalized direction tuning curves, all significantly tuned units")
ax_mat.set_xticks(orientations_deg)
plt.colorbar(im, ax=ax_mat, label="Normalized rate", fraction=0.03, pad=0.02)

plt.savefig("fig04_population_summary.png", dpi=150)
plt.close()

## Results

Across the population of well-isolated visual-cortex units in this session, a large
fraction show statistically significant tuning for grating direction (one-way ANOVA,
p < 0.01), and the distribution of orientation selectivity indices (gOSI) is shifted well
above the value expected from an untuned (flat) response (gOSI approx 0). The example unit's
raster (Figure 2) shows a clean, repeatable increase in firing confined to a pair of
opposite-direction trial blocks (i.e., a single stimulus orientation), and its PSTHs
(Figure 3) confirm a fast-onset, sustained response for the preferred direction with weak
or absent responses to orthogonal directions. The population tuning matrix (Figure 4,
bottom) shows that preferred orientations are distributed across the full range of stimulus
directions and that most tuned units have a single, well-defined tuning peak, the classic
signature of orientation-selective visual cortical neurons.

print("Analysis complete. Figures and CSV outputs written to the working directory.")